In [ ]:
#图3及图4
import pandas as pd
import os
import scipy.stats as st
import numpy as np
import statsmodels.stats.multitest as smt
import seaborn as sns
import matplotlib.pyplot as plt
from statannotations.Annotator import Annotator
import re
import scipy.stats as stats
from statsmodels.stats.multitest import fdrcorrection
#  辅助函数：计算生态位宽度 (Bj) 
def calculate_levins_bj(df):
    """
    计算每个物种的 Levin's 生态位宽度 Bj。
    
    公式: Bj = 1 / sum(Pij^2)
    其中 Pij 为物种 j 在样本 i 中的相对丰度。
    
    参数:
        df (pd.DataFrame): 行 = OTU/物种, 列 = 样本, 值为相对丰度
    
    返回:
        pd.Series: 每个物种对应的 Bj 值
    """
    matrix = df.astype(float)
    
    # 对每个物种（行），跨所有样本计算 Pij² 的加和
    sum_sq = (matrix ** 2).sum(axis=1)
    
    # 避免除以零，将0替换为NaN后再填回0
    bj = 1.0 / sum_sq.replace(0, np.nan)
    
    return bj.fillna(0)


def classify_generalist_specialist(
    relative_abundance_matrix,
    num_permutations=1000,
    seed=42
):
    """
    基于生态位宽度（Levin's Bj）的置换检验，将物种分类为：
    广适种（Generalist）、专适种（Specialist）或中性种（Neutral）。
    
    分类标准：
        Generalist : 观测 Bj 显著大于零分布（p_generalist < 0.05）
        Specialist : 观测 Bj 显著小于零分布（p_specialist < 0.05）
        Neutral    : 其余物种
    
    参数:
        relative_abundance_matrix (pd.DataFrame):
            行 = OTU/物种, 列 = 样本/生境, 值为相对丰度
        num_permutations (int):
            置换次数，默认 1000 次
        seed (int):
            随机种子，保证结果可重复，默认 42
    
    返回:
        pd.DataFrame: index = 物种名，包含以下列：
            - Bj_Observed  : 观测生态位宽度
            - Bj_Expected  : 零分布的平均生态位宽度
            - Bj_Std       : 零分布的标准差
            - Z_Score      : 标准化得分，衡量偏离零分布的程度
            - p_generalist : 右尾 p 值（观测值是否显著偏大）
            - p_specialist : 左尾 p 值（观测值是否显著偏小）
            - Ecotype      : 分类结果（Generalist / Specialist / Neutral）
    """
    # ---------- 1. 数据准备 ----------
    mat = relative_abundance_matrix.fillna(0).astype(float)
    species = mat.index
    n_species, n_samples = mat.shape
    
    print(f"输入矩阵维度: {n_species} 个物种 × {n_samples} 个样本")

    # ---------- 2. 计算观测 Bj ----------
    observed_bj = calculate_levins_bj(mat)

    # ---------- 3. 置换检验，构建零分布 ----------
    # 使用独立随机生成器，避免影响全局随机状态
    rng = np.random.default_rng(seed=seed)
    
    # 预分配结果数组：行 = 物种，列 = 每次置换的 Bj
    permuted_bj_array = np.zeros((n_species, num_permutations), dtype=float)

    for i in range(num_permutations):
        # 将整个相对丰度矩阵展平后随机打乱
        # 目的：彻底破坏物种与样本之间的对应关系，生成零假设下的分布
        shuffled_flat = mat.values.flatten().copy()
        rng.shuffle(shuffled_flat)
        
        # 重塑回原始矩阵形状，计算该置换下每个物种的 Bj
        shuffled_df = pd.DataFrame(
            shuffled_flat.reshape(n_species, n_samples),
            index=species,
            columns=mat.columns
        )
        permuted_bj_array[:, i] = calculate_levins_bj(shuffled_df).values

    # ---------- 4. 零分布统计量 ----------
    # 零分布的均值：代表随机期望下的 Bj
    expected_bj = permuted_bj_array.mean(axis=1)
    
    # 零分布的标准差（使用无偏估计 ddof=1）
    std_null = permuted_bj_array.std(axis=1, ddof=1)
    
    # Z-score：观测值偏离零分布均值的标准化程度
    # 标准差为 0 时（所有置换结果相同）设为 NaN，避免除以零
    z_score = (observed_bj.values - expected_bj) / np.where(
        std_null == 0, np.nan, std_null
    )

    # ---------- 5. 计算置换 p 值 ----------
    # 使用 +1 修正（Phipson & Smyth 2010），避免 p 值为 0 的情况
    obs_val = observed_bj.values.reshape(-1, 1)
    
    # 右尾 p 值：零分布中 Bj >= 观测值的比例
    # p 值越小，说明观测 Bj 显著偏大 → 倾向于 Generalist
    p_gen = (
        (permuted_bj_array >= obs_val).sum(axis=1) + 1
    ) / (num_permutations + 1)
    
    # 左尾 p 值：零分布中 Bj <= 观测值的比例
    # p 值越小，说明观测 Bj 显著偏小 → 倾向于 Specialist
    p_spe = (
        (permuted_bj_array <= obs_val).sum(axis=1) + 1
    ) / (num_permutations + 1)

    # ---------- 6. 汇总结果 ----------
    results = pd.DataFrame({
        'Bj_Observed' : observed_bj.values,
        'Bj_Expected' : expected_bj,
        'Bj_Std'      : std_null,
        'Z_Score'     : z_score,
        'p_generalist': p_gen,
        'p_specialist': p_spe,
    }, index=species)

    # ---------- 7. 物种分类 ----------
    results['Ecotype'] = 'Neutral'
    results.loc[results['p_generalist'] < 0.05, 'Ecotype'] = 'Generalist'
    results.loc[results['p_specialist'] < 0.05, 'Ecotype'] = 'Specialist'

    # ---------- 8. 输出统计摘要 ----------
    print("\n物种分类统计:")
    print(results['Ecotype'].value_counts().to_string())

    return results


#  细菌生态型
# 输入数据准备 (Input data prep)
# 填充颜色矩阵 (Populating color matrix)
# 根据门 (Phylum) 进行颜色编码
# 读取包含分类信息的CSV文件
metadata = pd.read_csv("/mnt/d/study/master/metadata.tsv", sep='\t')
bacteria_ASV = pd.read_csv("/mnt/d/study/master/meiji/bacteria_ASV.csv", sep=",", header=0)
# 构建要选择的列名列表，包括 "ASV" 和所有的样本ID
sample_ids = metadata['Sample ID'].tolist()
columns_to_select = ["ASV"] + sample_ids
# 从 bacteria_ASV DataFrame 中选择指定的列
b_ASV = bacteria_ASV[columns_to_select]
# 将 'ASV' 列设置为 DataFrame 的索引 (行名)
b_ASV.set_index(keys="ASV", inplace=True)
b_tax = bacteria_ASV.iloc[:, 1:9].copy()
b_tax.set_index(keys="ASV", inplace=True)
print(len(b_ASV))
# 计算 ASV 相对丰度
b_ASV_df = b_ASV.copy()

#按种
b_species = b_ASV.join(b_tax['Species'])
# 按照 'Species' 列进行分组，并对每个样本的丰度求和
b_species = b_species.groupby('Species')[sample_ids].sum()
#排序
b_species = b_species.sort_index()
b_species_df = b_species.copy()
print(len(b_species_df))
# 将当前的 ASV 索引重置为一个普通列。
b_tax_reset_index = b_tax.reset_index()
# 将 'Species' 列设置为新的索引。
b_tax_species = b_tax_reset_index.groupby('Species').first()
b_tax_species = b_tax_species.drop(columns=['ASV'])
b_tax_species = b_tax_species.sort_index()
# 进行替换操作
b_tax_species['Phylum'] = b_tax_species['Phylum'].replace('p__unclassified_k__norank_d__Bacteria', 'Unclassified')
b_tax_species['Phylum'] = b_tax_species['Phylum'].replace('p__SAR324_cladeMarine_group_B', 'SAR324')
# 去除 'p__' 前缀
b_tax_species['Phylum'] = b_tax_species['Phylum'].astype(str).str.replace("p__", "")
# 获取所有唯一的物种学名
b_unique_species_names = b_species.index.unique()
# 创建一个从物种学名到 'species_数字' 格式的映射字典
b_species_to_numeric_id = {
    species_name: f"species_{idx + 1}"
    for idx, species_name in enumerate(b_unique_species_names.sort_values()) # 先排序确保一致性
}
# 转换 b_species 的索引
b_species = b_species.rename(index= b_species_to_numeric_id)
# 转换 b_tax_species 的索引
b_tax_species = b_tax_species.rename(index= b_species_to_numeric_id)
b_species_df = b_species.copy()
print(len(b_species_df))

#核心species
# 在 Python 中，可以直接用 DataFrame 的除法功能，对列求和然后进行广播除法
b_species_rel = b_species_df.div(b_species_df.sum(axis=0), axis=1)
# 先把样本、origin 和 species 丰度组合成一个长表
# pandas 的 stack() 和 reset_index() 可以实现类似 melt 的功能
b_rel_long = b_species_rel.stack().reset_index()
b_rel_long.columns = ['species', 'Sample', 'value'] # 重命名列以匹配 R 的 melt 结果
# 创建从 Sample ID 到 Origin 和 Niche 的映射字典
sample_origins_map = metadata.set_index('Sample ID')['Origin'].to_dict()
sample_niches_map = metadata.set_index('Sample ID')['Niche'].to_dict()
# 将信息添加到长表中
b_rel_long['Origin'] = b_rel_long['Sample'].map(sample_origins_map)
b_rel_long['Niche'] = b_rel_long['Sample'].map(sample_niches_map)
b_species_total_counts = b_species.sum(axis=1) # 计算每个 species 在所有样本中的总丰度
b_total_counts_across_all_species = b_species_total_counts.sum() # 计算所有 species 的总丰度
b_species_relative_abundance = b_species_total_counts / b_total_counts_across_all_species
# 定义整体相对丰度的阈值 (0.1% 转换为小数)
overall_abundance_threshold = 0.001
# 筛选出满足整体丰度阈值的 species 名称列表
b_species_above_overall_threshold = b_species_relative_abundance[b_species_relative_abundance >= overall_abundance_threshold].index.tolist()
# 现在，只保留那些通过了整体丰度筛选的 species，用于后续的居群出现次数计算
b_rel_long_filtered_by_overall_abundance = b_rel_long[b_rel_long['species'].isin(b_species_above_overall_threshold)]
#计算 species 在每个 Origin 中是否出现 (平均丰度 > 0) 
b_species_origin_mean_filtered = b_rel_long_filtered_by_overall_abundance.groupby(['species', 'Origin'])['value'].mean().reset_index()
b_species_origin_mean_filtered.rename(columns={'value': 'mean_abundance_in_origin'}, inplace=True)
# species 在某个 Origin 中出现，如果其在该 Origin 中的平均丰度大于 0
b_species_in_origin = b_species_origin_mean_filtered[b_species_origin_mean_filtered['mean_abundance_in_origin'] > 0]
b_species_origin_count = b_species_in_origin.groupby('species')['Origin'].count().reset_index(name='origin_count')
#计算 species 在每个 Niche 中是否出现 (平均丰度 > 0) 
b_species_niche_mean_filtered = b_rel_long_filtered_by_overall_abundance.groupby(['species', 'Niche'])['value'].mean().reset_index()
b_species_niche_mean_filtered.rename(columns={'value': 'mean_abundance_in_niche'}, inplace=True)
# species 在某个 Niche 中出现，如果其在该 Niche 中的平均丰度大于 0
b_species_in_niche = b_species_niche_mean_filtered[b_species_niche_mean_filtered['mean_abundance_in_niche'] > 0]
b_species_niche_count = b_species_in_niche.groupby('species')['Niche'].count().reset_index(name='niche_count')
# 筛选核心 species 
# 合并 Origin 和 Niche 出现次数
b_species_counts_combined = pd.merge(b_species_origin_count, b_species_niche_count, on='species', how='inner')
# 定义出现次数阈值
origin_occurrence_threshold = 3
niche_occurrence_threshold = 2
# 筛选满足所有条件的 species
b_core_species_names = b_species_counts_combined[
    (b_species_counts_combined['origin_count'] >= origin_occurrence_threshold) &
    (b_species_counts_combined['niche_count'] >= niche_occurrence_threshold)
]['species'].tolist()
# 提取核心 species 的相对丰度数据
b_core_abundance = b_species_rel.loc[b_core_species_names]
b_core_species_raw_counts = b_species.loc[b_core_species_names]
print(len(b_core_species_raw_counts))

#最特殊species
#计算所有 species 整体相对丰度的中位数 
b_median_relative_abundance = b_species_relative_abundance.median()
print(b_median_relative_abundance)
# 定义稀有类群的丰度阈值 
b_rare_threshold = 0.00001
# 筛选出稀有 species 
b_rare_species = b_species_relative_abundance[b_species_relative_abundance < b_rare_threshold]
# 获取稀有 species 的名称（索引）
b_rare_species_names = b_rare_species.index.tolist()
len(b_rare_species_names)
b_species_db_transposed = b_species_df.T 
b_speciesdfmelt = b_species_db_transposed.stack().reset_index()# 长宽表格转换
b_speciesdfmelt.columns = ['Sample', 'species', 'value']
b_speciesdfmelt = b_speciesdfmelt[b_speciesdfmelt['value'] > 0]
sample_group_map = metadata.set_index('Sample ID')['Group'].to_dict()
b_control_species = b_speciesdfmelt.copy()
b_control_species['Group'] = b_control_species['Sample'].map(sample_group_map)# 添加 Group 信息
b_control_species.dropna(subset=['Group'], inplace=True) # 移除 Group 为 NaN 的行
# 提取 Group 和 species 列并去重
b_controlspecies_unique = b_control_species[['Group', 'species']].drop_duplicates()
# 统计每个 species 出现在多少个 Group 中
b_controlspecies_unique = b_controlspecies_unique.groupby('species').size().reset_index(name='group_count') 
# 只保留那些只在一个 Group 中出现的 species
b_controlspecies_unique = b_controlspecies_unique[b_controlspecies_unique['group_count'] == 1] 
# 获取只在一个 Group 中出现的 species 名称列表
b_unique_species_names_single_group = b_controlspecies_unique['species'].tolist()
# 将两个列表转换为集合，以便快速找到交集
b_set_rare_species = set(b_rare_species_names)
b_set_unique_species_single_group = set(b_unique_species_names_single_group) # 使用这个变量
# 找到同时满足两个条件的 species (即两个集合的交集)
b_rare_unique_species_names = list(b_set_rare_species.intersection(b_set_unique_species_single_group))
#提取这些“最特殊稀有 species”的原始丰度数据
b_rare_unique_abundance = b_species_df.loc[b_rare_unique_species_names]
b_rare_unique_species_raw_counts = b_species.loc[b_rare_unique_species_names]
print(len(b_rare_unique_species_raw_counts))

#生态位广适类群和专适类群
print(" 开始生态位广适/专适类群识别 ")
# 执行生态位宽度置换检验与 Z-score 计算
num_permutations = 1000 
b_niche_breadth_results = classify_generalist_specialist(
    b_species_rel, num_permutations=num_permutations
)
print("\n 生态位宽度置换检验结果 (含 Z-score, 前5个 species) ")
print(b_niche_breadth_results.head())
b_generalists_species = b_niche_breadth_results[b_niche_breadth_results['Ecotype'] == 'Generalist'].index.tolist()
b_specialists_species = b_niche_breadth_results[b_niche_breadth_results['Ecotype'] == 'Specialist'].index.tolist()
print(f"\n识别出 **{len(b_generalists_species)}** 个广适类群 species (Z-score >= 2)。")
print(f"识别出 **{len(b_specialists_species)}** 个专适类群 species (Z-score <= -2)。")
print("\n 广适类群 species 的相对丰度数据 (前5行，前5列) ")
if not b_generalists_species:
    print("无广适类群 species 数据。")
else:
    b_generalists_raw_counts = b_species.loc[b_generalists_species]
    b_generalists_abundance = b_species_rel.loc[b_generalists_species]
    print(b_generalists_abundance.head())
print("\n 专适类群 species 的相对丰度数据 (前5行，前5列) ")
if not b_specialists_species:
    print("无专适类群 species 数据。")
else:
    b_specialists_raw_counts = b_species.loc[b_specialists_species]
    b_specialists_abundance = b_species_rel.loc[b_specialists_species]
print(b_specialists_abundance.head())
print("\n 生态位广适/专适类群识别完成 ")

# 加载相对丰度、丰度物种、稀有物种、广适种和狭适种数据
b_df_OTU_rel = b_species_rel 
b_df_abun = b_core_abundance 
b_df_rare = b_rare_unique_abundance
b_df_gen = b_generalists_abundance
b_df_spe = b_specialists_abundance
# 加载生态系统数据
b_ecosystem = metadata
b_ecosystem = b_ecosystem.iloc[:, :2]
b_ecosystem.set_index(b_ecosystem.columns[0], inplace=True)
# 绘制分组的平均相对丰度 vs. 占据的位点数量的散点图
# 准备用于散点图的输入 DataFrame，包括平均相对丰度、占据的位点数量和分组
# 获取丰度物种、稀有物种、广适种和狭适种的OTU索引列表
b_abun_OTU = list(b_df_abun.index)
b_rare_OTU = list(b_df_rare.index)
b_gen_OTU = list(b_df_gen.index)
b_spe_OTU = list(b_df_spe.index)
# 计算每个OTU的平均相对丰度
# axis=1 表示对每一行（即每个OTU）进行操作
b_relabun = b_df_OTU_rel.mean(axis=1)
# 将 Series 转换为 DataFrame，并命名列为 'meanRelabun'
b_relabun = b_relabun.to_frame(name='meanRelabun')
# 计算每个OTU占据的位点数量（即在多少个样本中丰度不为0）
# iloc[:, 1:] 假设第一列是索引或OTU名称，我们只计算样本列
# != 0 检查每个值是否不为0，返回布尔值DataFrame
# sum(axis=1) 计算每行（每个OTU）的True（即不为0的位点）的数量
b_sites = (b_df_OTU_rel.iloc[:, 1:] != 0).sum(axis=1).to_frame(name='sitesCount')
# 将平均相对丰度 DataFrame 和位点数量 DataFrame 根据索引合并
b_relabun_sites = pd.merge(b_relabun, b_sites, left_index=True, right_index=True)
# 定义一个字典，将分组标签与其对应的OTU索引列表关联起来
b_lists = {'Cores': b_abun_OTU,
         'Uniques': b_rare_OTU,
         'Generalists': b_gen_OTU,
         'Specialists': b_spe_OTU}
# 在 b_relabun_sites DataFrame 中添加一个名为 'group' 的新列，用于存储分组标签
b_relabun_sites['group'] = ''
# 遍历定义的分组和其OTU索引列表
for label, indices in b_lists.items():
    # 根据OTU索引判断，将对应的分组标签赋给 'group' 列
    b_relabun_sites.loc[b_relabun_sites.index.isin(indices), 'group'] = label
# 过滤掉 'group' 列为空（即不属于任何已知分组）的行
b_relabun_sites = b_relabun_sites[b_relabun_sites['group'].astype(bool)]
# 按照自定义的顺序对 'group' 列进行排序，确保图例和绘图的顺序一致
b_custom_order = ["Cores", "Uniques", "Generalists", "Specialists"]
b_relabun_sites['group'] = pd.Categorical(b_relabun_sites['group'], categories= b_custom_order, ordered=True)
b_relabun_sites_sorted = b_relabun_sites.sort_values(by='group')
# 创建一个散点图
# make a scatter plot
# 定义分组与颜色之间的映射关系
b_group_colors = {
    'Cores': '#5499C7',  # 蓝色
    'Uniques': '#D4AC0D',      # 金色
    'Generalists': '#EC7063',    # 红色
    'Specialists': '#45B39D'     # 绿色
}
# 设置图表的整体尺寸 (宽度, 高度)
plt.rcParams["figure.figsize"] = (4, 4) # 注意：plt.rcParams 应在 plt.figure() 或 sns.scatterplot() 之前设置，
                                     # 或者使用 plt.figure(figsize=(4,4)) 来创建图
# 使用 seaborn 库创建散点图
# data: 要绘图的 DataFrame
# x: X轴数据 ('sitesCount' - 占据的位点数量)
# y: Y轴数据 ('meanRelabun' - 平均相对丰度)
# hue: 根据 'group' 列的值给散点上色，并自动生成图例
# palette: 使用预定义的颜色映射
# s: 散点的大小
# linewidth: 散点边缘线的宽度
# edgecolor: 散点边缘线的颜色
# alpha: 散点的透明度
b_ax = sns.scatterplot(data= b_relabun_sites_sorted, x='sitesCount', y='meanRelabun', hue='group',
                     palette= b_group_colors, s=100, linewidth=0.5, edgecolor='black', alpha=0.6)
# 将Y轴（平均相对丰度）设置为对数刻度，以便更好地展示大范围的丰度数据
plt.yscale('log')
# 设置X轴和Y轴的标签及其字体大小
b_ax.set_xlabel('Samples occupied', size=13)
b_ax.set_ylabel('Mean relative abundance', size=13)
# 设置图表的标题及其字体大小
b_ax.set_title('Bacterial ecotype distribution', size=13)
# 显示图例 (由 hue 参数自动生成)
b_ax.legend()
# 定义输出目录（确保该目录存在，如果不存在，os.makedirs 会创建它）
output_dir = '/mnt/d/study/master/Extended_Data_tables/Figure_3/'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"创建输出目录: {output_dir}")
# 保存图表
plt.savefig(os.path.join(output_dir, 'ecotype_distribution_bacteria.png'), bbox_inches='tight', dpi=600)
plt.show()

#  数据准备：不同分组在生态系统中的分布 
# 将各个分组（丰度物种、稀有物种、广适种、狭适种）的DataFrame进行转置
# 转置后，行是样本，列是OTU
b_df_abun_T = b_df_abun.T
# 计算每个样本中（即每行）非零OTU的数量（即该样本包含的该分组OTU的数量）
b_df_abun_ct = b_df_abun_T.apply(lambda row: np.count_nonzero(row), axis=1).to_frame(name='Count')
# 为这些计数添加一个 'group' 列，标记为 'Cores'
b_df_abun_ct['ecotype'] = 'Cores'
# 将计数DataFrame与生态系统元数据合并，基于它们的索引（样本ID）
b_df_abun_eco = pd.merge(b_df_abun_ct, b_ecosystem, left_index = True, right_index = True)
# 对稀有物种重复上述步骤
b_df_rare_T = b_df_rare.T
b_df_rare_ct = b_df_rare_T.apply(lambda row: np.count_nonzero(row), axis=1).to_frame(name='Count')
b_df_rare_ct['ecotype'] = 'Uniques'
b_df_rare_eco = pd.merge(b_df_rare_ct, b_ecosystem, left_index = True, right_index = True)
# 对广适种重复上述步骤
b_df_gen_T = b_df_gen.T
b_df_gen_ct = b_df_gen_T.apply(lambda row: np.count_nonzero(row), axis=1).to_frame(name='Count')
b_df_gen_ct['ecotype'] = 'Generalists'
b_df_gen_eco = pd.merge(b_df_gen_ct, b_ecosystem, left_index = True, right_index = True)
# 对狭适种重复上述步骤
b_df_spe_T = b_df_spe.T
b_df_spe_ct = b_df_spe_T.apply(lambda row: np.count_nonzero(row), axis=1).to_frame(name='Count')
b_df_spe_ct['ecotype'] = 'Specialists'
b_df_spe_eco = pd.merge(b_df_spe_ct, b_ecosystem, left_index = True, right_index = True)
# 将所有分组的数据（丰度物种、稀有物种、广适种、狭适种）纵向合并到一个大的DataFrame中
b_df_ecotype_eco = pd.concat([b_df_abun_eco, b_df_rare_eco, b_df_gen_eco, b_df_spe_eco])
#  绘图参数设置 
# 定义箱线图的X轴、Y轴和分组（颜色区分）变量
x = "ecotype" # X轴：分组
y = "Count" # Y轴：OTU数量
hue = "Group" # 分组（颜色）：生态系统类型
# 定义 hue 变量（生态系统类型）的显示顺序，以便在图例和绘图中保持一致
b_hue_order = ["JRG", "JJG", "TZG", "PAG", "JRN", "JJN", "TZN", "PAN"]
# 定义 X 轴上分组（group）的显示顺序
b_order = ["Cores", "Uniques", "Generalists", "Specialists"]
# 定义生态系统类型与颜色之间的映射关系
b_group_colors = {
   "JRG": "#8C57A2FF",
   "JJG": "#3EBCB6",
   "TZG": "#82581FFF",
   "PAG": "#2F509EFF",
   "JRN": "#E5614CFF",
   "JJN": "#97A1A7FF",
   "TZN": "#DC9445FF",
   "PAN": "#bee183"
}
# 定义要进行统计比较的配对列表
# 每个元组包含两个要比较的组，格式为 ((group_x, hue_x), (group_y, hue_y))
# 这些是在FDR校正后仍然显著的比较对，未显著的已被注释掉
pairs = [
    # Cores (核心物种) 的比较对 (共28对)
    (("Cores", "JRG"), ("Cores", "JJG")),
    (("Cores", "JRG"), ("Cores", "TZG")),
    (("Cores", "JRG"), ("Cores", "PAG")),
    (("Cores", "JRG"), ("Cores", "JRN")),
    (("Cores", "JRG"), ("Cores", "JJN")),
    (("Cores", "JRG"), ("Cores", "TZN")),
    (("Cores", "JRG"), ("Cores", "PAN")),
    (("Cores", "JJG"), ("Cores", "TZG")),
    (("Cores", "JJG"), ("Cores", "PAG")),
    (("Cores", "JJG"), ("Cores", "JRN")),
    (("Cores", "JJG"), ("Cores", "JJN")),
    (("Cores", "JJG"), ("Cores", "TZN")),
    (("Cores", "JJG"), ("Cores", "PAN")),
    (("Cores", "TZG"), ("Cores", "PAG")),
    (("Cores", "TZG"), ("Cores", "JRN")),
    (("Cores", "TZG"), ("Cores", "JJN")),
    (("Cores", "TZG"), ("Cores", "TZN")),
    (("Cores", "TZG"), ("Cores", "PAN")),
    (("Cores", "PAG"), ("Cores", "JRN")),
    (("Cores", "PAG"), ("Cores", "JJN")),
    (("Cores", "PAG"), ("Cores", "TZN")),
    (("Cores", "PAG"), ("Cores", "PAN")),
    (("Cores", "JRN"), ("Cores", "JJN")),
    (("Cores", "JRN"), ("Cores", "TZN")),
    (("Cores", "JRN"), ("Cores", "PAN")),
    (("Cores", "JJN"), ("Cores", "TZN")),
    (("Cores", "JJN"), ("Cores", "PAN")),
    (("Cores", "TZN"), ("Cores", "PAN")),

    # Uniques (稀有物种) 的比较对 (共28对)
    (("Uniques", "JRG"), ("Uniques", "JJG")),
    (("Uniques", "JRG"), ("Uniques", "TZG")),
    (("Uniques", "JRG"), ("Uniques", "PAG")),
    (("Uniques", "JRG"), ("Uniques", "JRN")),
    (("Uniques", "JRG"), ("Uniques", "JJN")),
    (("Uniques", "JRG"), ("Uniques", "TZN")),
    (("Uniques", "JRG"), ("Uniques", "PAN")),
    (("Uniques", "JJG"), ("Uniques", "TZG")),
    (("Uniques", "JJG"), ("Uniques", "PAG")),
    (("Uniques", "JJG"), ("Uniques", "JRN")),
    (("Uniques", "JJG"), ("Uniques", "JJN")),
    (("Uniques", "JJG"), ("Uniques", "TZN")),
    (("Uniques", "JJG"), ("Uniques", "PAN")),
    (("Uniques", "TZG"), ("Uniques", "PAG")),
    (("Uniques", "TZG"), ("Uniques", "JRN")),
    (("Uniques", "TZG"), ("Uniques", "JJN")),
    (("Uniques", "TZG"), ("Uniques", "TZN")),
    (("Uniques", "TZG"), ("Uniques", "PAN")),
    (("Uniques", "PAG"), ("Uniques", "JRN")),
    (("Uniques", "PAG"), ("Uniques", "JJN")),
    (("Uniques", "PAG"), ("Uniques", "TZN")),
    (("Uniques", "PAG"), ("Uniques", "PAN")),
    (("Uniques", "JRN"), ("Uniques", "JJN")),
    (("Uniques", "JRN"), ("Uniques", "TZN")),
    (("Uniques", "JRN"), ("Uniques", "PAN")),
    (("Uniques", "JJN"), ("Uniques", "TZN")),
    (("Uniques", "JJN"), ("Uniques", "PAN")),
    (("Uniques", "TZN"), ("Uniques", "PAN")),

    # Generalists (广适种) 的比较对 (共28对)
    (("Generalists", "JRG"), ("Generalists", "JJG")),
    (("Generalists", "JRG"), ("Generalists", "TZG")),
    (("Generalists", "JRG"), ("Generalists", "PAG")),
    (("Generalists", "JRG"), ("Generalists", "JRN")),
    (("Generalists", "JRG"), ("Generalists", "JJN")),
    (("Generalists", "JRG"), ("Generalists", "TZN")),
    (("Generalists", "JRG"), ("Generalists", "PAN")),
    (("Generalists", "JJG"), ("Generalists", "TZG")),
    (("Generalists", "JJG"), ("Generalists", "PAG")),
    (("Generalists", "JJG"), ("Generalists", "JRN")),
    (("Generalists", "JJG"), ("Generalists", "JJN")),
    (("Generalists", "JJG"), ("Generalists", "TZN")),
    (("Generalists", "JJG"), ("Generalists", "PAN")),
    (("Generalists", "TZG"), ("Generalists", "PAG")),
    (("Generalists", "TZG"), ("Generalists", "JRN")),
    (("Generalists", "TZG"), ("Generalists", "JJN")),
    (("Generalists", "TZG"), ("Generalists", "TZN")),
    (("Generalists", "TZG"), ("Generalists", "PAN")),
    (("Generalists", "PAG"), ("Generalists", "JRN")),
    (("Generalists", "PAG"), ("Generalists", "JJN")),
    (("Generalists", "PAG"), ("Generalists", "TZN")),
    (("Generalists", "PAG"), ("Generalists", "PAN")),
    (("Generalists", "JRN"), ("Generalists", "JJN")),
    (("Generalists", "JRN"), ("Generalists", "TZN")),
    (("Generalists", "JRN"), ("Generalists", "PAN")),
    (("Generalists", "JJN"), ("Generalists", "TZN")),
    (("Generalists", "JJN"), ("Generalists", "PAN")),
    (("Generalists", "TZN"), ("Generalists", "PAN")),

    # Specialists (狭适种) 的比较对 (共28对)
    (("Specialists", "JRG"), ("Specialists", "JJG")),
    (("Specialists", "JRG"), ("Specialists", "TZG")),
    (("Specialists", "JRG"), ("Specialists", "PAG")),
    (("Specialists", "JRG"), ("Specialists", "JRN")),
    (("Specialists", "JRG"), ("Specialists", "JJN")),
    (("Specialists", "JRG"), ("Specialists", "TZN")),
    (("Specialists", "JRG"), ("Specialists", "PAN")),
    (("Specialists", "JJG"), ("Specialists", "TZG")),
    (("Specialists", "JJG"), ("Specialists", "PAG")),
    (("Specialists", "JJG"), ("Specialists", "JRN")),
    (("Specialists", "JJG"), ("Specialists", "JJN")),
    (("Specialists", "JJG"), ("Specialists", "TZN")),
    (("Specialists", "JJG"), ("Specialists", "PAN")),
    (("Specialists", "TZG"), ("Specialists", "PAG")),
    (("Specialists", "TZG"), ("Specialists", "JRN")),
    (("Specialists", "TZG"), ("Specialists", "JJN")),
    (("Specialists", "TZG"), ("Specialists", "TZN")),
    (("Specialists", "TZG"), ("Specialists", "PAN")),
    (("Specialists", "PAG"), ("Specialists", "JRN")),
    (("Specialists", "PAG"), ("Specialists", "JJN")),
    (("Specialists", "PAG"), ("Specialists", "TZN")),
    (("Specialists", "PAG"), ("Specialists", "PAN")),
    (("Specialists", "JRN"), ("Specialists", "JJN")),
    (("Specialists", "JRN"), ("Specialists", "TZN")),
    (("Specialists", "JRN"), ("Specialists", "PAN")),
    (("Specialists", "JJN"), ("Specialists", "TZN")),
    (("Specialists", "JJN"), ("Specialists", "PAN")),
    (("Specialists", "TZN"), ("Specialists", "PAN"))
]
#  提取 P 值并执行 FDR 校正 
# 存储原始 P 值
p_values = []
# 存储每个 P 值对应的比较对，以便后续追踪
comparison_labels = []
# 对每一对进行循环
for pair in pairs:
    ecotype1, hue1 = pair[0]
    ecotype2, hue2 = pair[1]
    # 从 DataFrame 中筛选出第一组的数据
    data1 = b_df_ecotype_eco[(b_df_ecotype_eco['ecotype'] == ecotype1) &
                           (b_df_ecotype_eco['Group'] == hue1)]['Count']
    # 从 DataFrame 中筛选出第二组的数据
    data2 = b_df_ecotype_eco[(b_df_ecotype_eco['ecotype'] == ecotype2) &
                           (b_df_ecotype_eco['Group'] == hue2)]['Count']
    # 确保两组都有数据才能进行比较
    if len(data1) > 1 and len(data2) > 1: # 通常需要至少2个样本才能进行T检验
        # 执行 Mann-Whitney U 检验（非参数，对数据分布没有严格要求）
        # 如果您确定数据服从正态分布且方差齐性，可以使用 stats.ttest_ind
        statistic, p_val = stats.mannwhitneyu(data1, data2, alternative='two-sided')
        p_values.append(p_val)
        comparison_labels.append(f"{ecotype1}-{hue1} vs {ecotype2}-{hue2}")
    else:
        # 如果数据不足，则跳过或记录为 NaN
        p_values.append(np.nan)
        comparison_labels.append(f"{ecotype1}-{hue1} vs {ecotype2}-{hue2} (数据不足)")
# 过滤掉由于数据不足而产生的 NaN P 值
valid_p_values = [p for p in p_values if not np.isnan(p)]
valid_comparison_labels = [label for p, label in zip(p_values, comparison_labels) if not np.isnan(p)]
if len(valid_p_values) == 0:
    print("没有足够的有效比较可以执行 FDR 校正。请检查您的数据和比较对。")
else:
    # 执行 FDR 校正 (Benjamini-Hochberg 方法)
    # alpha 是您设定的显著性水平，例如 0.05
    reject, pvals_corrected = fdrcorrection(pvals=valid_p_values, alpha=0.05, method='indep')
    # 结果展示 
    print("\n FDR 校正结果 (Benjamini-Hochberg) ")
    print(f"原始有效比较数量: {len(valid_p_values)}")
    print(f"FDR 校正后的显著性水平 (alpha): 0.05")
    significant_pairs_after_fdr = []
    insignificant_pairs_after_fdr = []
    for i, label in enumerate(valid_comparison_labels):
        original_p = valid_p_values[i]
        corrected_p = pvals_corrected[i]
        is_significant = reject[i]
        result_string = f"比较: {label}, 原始 P 值: {original_p:.4f}, 校正后 P 值 (q 值): {corrected_p:.4f}, 是否显著: {is_significant}"
        print(result_string)
        if is_significant:
            significant_pairs_after_fdr.append(label)
        else:
            insignificant_pairs_after_fdr.append(label)
    print("\n 显著的比较对 (FDR 校正后) ")
    if significant_pairs_after_fdr:
        for p in significant_pairs_after_fdr:
            print(p)
    else:
        print("FDR 校正后没有发现显著的比较对。")
    print("\n 不显著的比较对 (FDR 校正后) ")
    if insignificant_pairs_after_fdr:
        for p in insignificant_pairs_after_fdr:
            print(p)
    else:
        print("所有比较对在 FDR 校正后均显著。")
b_pairs = [
    (('Cores', 'JRG'), ('Cores', 'JJG')),
    (('Cores', 'JRG'), ('Cores', 'TZG')),
    (('Cores', 'JRG'), ('Cores', 'PAN')),
    (('Cores', 'JJG'), ('Cores', 'JRN')),
    (('Cores', 'JJG'), ('Cores', 'JJN')),
    (('Cores', 'JJG'), ('Cores', 'PAN')),
    (('Cores', 'TZG'), ('Cores', 'JRN')),
    (('Cores', 'TZG'), ('Cores', 'JJN')),
    (('Cores', 'TZG'), ('Cores', 'PAN')),
    (('Cores', 'TZN'), ('Cores', 'PAN')),

    (('Uniques', 'TZG'), ('Uniques', 'PAG')),
    (('Uniques', 'TZN'), ('Uniques', 'PAN')),
    (('Uniques', 'JJG'), ('Uniques', 'PAG')),
    (('Uniques', 'TZG'), ('Uniques', 'JRN')),
    (('Uniques', 'JJG'), ('Uniques', 'JRN')),
    (('Uniques', 'JJG'), ('Uniques', 'JJN')),
    (('Uniques', 'JRG'), ('Uniques', 'JRN')),
    (('Uniques', 'JJG'), ('Uniques', 'PAN')),
    (('Uniques', 'TZG'), ('Uniques', 'PAN')),

    (('Generalists', 'TZG'), ('Generalists', 'PAG')),
    (('Generalists', 'JJN'), ('Generalists', 'TZN')),
    (('Generalists', 'TZG'), ('Generalists', 'JRN')),
    (('Generalists', 'JJG'), ('Generalists', 'PAG')),
    (('Generalists', 'PAG'), ('Generalists', 'JJN')),
    (('Generalists', 'JJG'), ('Generalists', 'JRN')),
    (('Generalists', 'TZG'), ('Generalists', 'JJN')),
    (('Generalists', 'JRG'), ('Generalists', 'JRN')),
    (('Generalists', 'JJG'), ('Generalists', 'JJN')),
    (('Generalists', 'TZG'), ('Generalists', 'TZN')),
    (('Generalists', 'JRG'), ('Generalists', 'JJN')),
    (('Generalists', 'JJG'), ('Generalists', 'TZN')),
    (('Generalists', 'TZG'), ('Generalists', 'PAN')),
    (('Generalists', 'JJG'), ('Generalists', 'PAN')),

    (('Specialists', 'TZG'), ('Specialists', 'PAG')),
    (('Specialists', 'JJN'), ('Specialists', 'TZN')),
    (('Specialists', 'TZN'), ('Specialists', 'PAN')),
    (('Specialists', 'TZG'), ('Specialists', 'JRN')),
    (('Specialists', 'JJG'), ('Specialists', 'PAG')),
    (('Specialists', 'JRN'), ('Specialists', 'TZN')),
    (('Specialists', 'JJG'), ('Specialists', 'JRN')),
    (('Specialists', 'TZG'), ('Specialists', 'JJN')),
    (('Specialists', 'JRG'), ('Specialists', 'JRN')),
    (('Specialists', 'JJG'), ('Specialists', 'JJN')),
    (('Specialists', 'TZG'), ('Specialists', 'TZN')),
    (('Specialists', 'JRG'), ('Specialists', 'JJN')),
    (('Specialists', 'JJG'), ('Specialists', 'TZN')),
    (('Specialists', 'TZG'), ('Specialists', 'PAN')),
    (('Specialists', 'JJG'), ('Specialists', 'PAN')),
    (('Specialists', 'JRG'), ('Specialists', 'PAN'))
]
# 设置 Matplotlib 图的尺寸（宽度，高度）
plt.rcParams["figure.figsize"] = (6, 6)
# 使用 Seaborn 绘制箱线图 (Box Plot)
b_ax = sns.boxplot(data=b_df_ecotype_eco, # 输入数据 DataFrame
                 x=x,               # X轴变量
                 y=y,               # Y轴变量
                 order=b_order,       # X轴分类的显示顺序
                 hue=hue,           # 根据哪个变量进行分组并上色
                 hue_order= b_hue_order, # hue变量的显示顺序
                 palette=b_group_colors,
                 legend=False) # 使用预定义的颜色映射
# 初始化 Annotator 对象，用于添加统计显著性注释
# 传入与 boxplot 相同的绘图参数，确保注释正确对齐
annot = Annotator(b_ax, b_pairs, data=b_df_ecotype_eco, x=x, y=y, order=b_order, hue=hue, hue_order= b_hue_order)
# 配置注释的测试方法和P值校正
# test='Mann-Whitney': 使用 Mann-Whitney U 检验进行非参数比较
# comparisons_correction="Benjamini-Hochberg": 使用 Benjamini-Hochberg 方法进行多重检验校正（FDR校正）
# verbose=2: 显示详细的测试结果信息
annot.configure(test='Mann-Whitney', comparisons_correction="Benjamini-Hochberg", verbose=2)
# 应用统计检验
annot.apply_test()
# 在图上绘制显著性标记（例如星号）
annot.annotate()
# 设置X轴标签为空字符串，因为组名本身已经很清晰
b_ax.set_xlabel('')
# 设置Y轴标签及其字体大小
b_ax.set_ylabel('Count', size=13)
# 设置图表的标题及其字体大小
b_ax.set_title('Number of bacterial species for ecotypes', size=13)
# 旋转X轴刻度标签，并调整字体大小和对齐方式，以防止重叠
plt.xticks(fontsize=13, rotation=45, ha='right')
# 调整图例的位置，使其位于图表外部的右侧中间位置
# title="Ecosystems": 设置图例标题
# bbox_to_anchor=(1.03, 0.35): 将图例锚点移动到轴外部 (1.03 表示X轴的103%位置，0.35 表示Y轴的35%位置)
#plt.legend(title="Ecosystems", bbox_to_anchor=(1.03, 0.35))
# 定义输出目录
output_dir = '/mnt/d/study/master/Extended_Data_tables/Figure_3/'
# 如果目录不存在，则创建它
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"已创建输出目录: {output_dir}")
# 保存图表
# bbox_inches='tight' 确保所有图表元素（包括图例和标签）都包含在保存的图像中
# dpi=600 设置图像的分辨率，提高清晰度
plt.savefig(os.path.join(output_dir, 'ecotype_ecosystems_bacteria.png'), bbox_inches='tight', dpi=600)
# 显示图表
plt.show()

#  计算分组所代表的门（Phyla）的比例 
# 为每个分组计算其所包含的门（Phylum）的观察比例
# 提取属于“丰度物种”的OTU的分类学信息
b_taxon = b_tax_species
# 定义模糊（Phylum）的模式（不区分大小写）
ambiguous_patterns = re.compile(
   r"uncultured|unclassified|norank",
   flags=re.IGNORECASE
)
# 统一处理函数，包含去除前缀和过滤模糊门
def get_filtered_phylum_proportion(df_group, taxon_df, ambiguous_pat):
    # 筛选出索引（OTU ID）存在于 df_group 中的分类学信息
    taxon_filtered = taxon_df[taxon_df.index.isin(df_group.index)].copy() # 使用 .copy() 避免 SettingWithCopyWarning
    # 去除 'p__' 前缀
    taxon_filtered['Phylum'] = taxon_filtered['Phylum'].astype(str).str.replace("p__", "")
    # 过滤掉匹配模糊模式的门
    # 使用 apply 和 lambda 函数来检查每个门名是否包含模糊模式
    taxon_filtered = taxon_filtered[~taxon_filtered['Phylum'].apply(lambda x: bool(ambiguous_pat.search(str(x))))]
    # 计算 'Phylum' 列中每个门的出现频率，并将其归一化为比例
    prop = taxon_filtered['Phylum'].value_counts(normalize=True).to_frame(name='Proportion')
    return prop
# 计算每个分组的门比例
b_prop_abun = get_filtered_phylum_proportion(b_df_abun, b_taxon, ambiguous_patterns)
b_prop_rare = get_filtered_phylum_proportion(b_df_rare, b_taxon, ambiguous_patterns)
b_prop_gen = get_filtered_phylum_proportion(b_df_gen, b_taxon, ambiguous_patterns)
b_prop_spe = get_filtered_phylum_proportion(b_df_spe, b_taxon, ambiguous_patterns)
#  绘制条形图显示每个分组的门比例 
# 设置 Matplotlib 图的默认尺寸（宽度，高度）
plt.rcParams["figure.figsize"] = (4, 4)
# 定义一个函数用于绘制单个分组的门比例条形图
def barplot_prop(df, groupname, color):
    """
    绘制给定DataFrame中每个门的比例条形图。
    参数:
        df (DataFrame): 包含 'Proportion' 列和 Phylum 作为索引的 DataFrame。
        groupname (str): 图表的标题和保存文件名的一部分，代表分组名称。
        color (str): 用于条形图的颜色。
    """
    # 使用 seaborn 绘制条形图
    # x = df.index: X轴为门的名称（DataFrame的索引）
    # y = df['Proportion']: Y轴为该门的比例
    # color = color: 设置条形图的颜色
    ax = sns.barplot(x = df.index, y = df['Proportion'], color = color)
    # 设置X轴标签、Y轴标签和图表标题
    ax.set_xlabel(' Bacterial phylum', size=13)
    ax.set_ylabel('Proportion', size=13)
    ax.set_title(groupname, size=13)
    # 旋转X轴刻度标签90度，并设置字体大小，防止标签重叠
    plt.xticks(fontsize=11, rotation = 90)
    # 定义保存图表的输出目录
    output_dir = '/mnt/d/study/master/Extended_Data_tables/Figure_3/'
    # 如果目录不存在，则创建它
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"已创建输出目录: {output_dir}")
    # 保存图表
    # bbox_inches='tight' 确保所有图表元素都包含在保存的图像中
    # dpi=600 设置图像的分辨率
    plt.savefig(os.path.join(output_dir, groupname + '_phylum_bar_bacteria.png'), bbox_inches='tight', dpi=600)
    # 显示图表
    plt.show()
# 调用函数为每个分组生成并保存条形图
barplot_prop(b_prop_abun, 'Cores', '#5499C7')   # 丰度物种
barplot_prop(b_prop_rare, 'Uniques', '#D4AC0D')       # 稀有物种#9
barplot_prop(b_prop_gen, 'Generalists', '#EC7063')      # 广适种
barplot_prop(b_prop_spe, 'Specialists', '#45B39D')      # 狭适种


#  真菌生态型
# 输入数据准备 (Input data prep)
# 填充颜色矩阵 (Populating color matrix)
# 根据门 (Phylum) 进行颜色编码
# 读取包含分类信息的CSV文件
metadata = pd.read_csv("/mnt/d/study/master/metadata.tsv", sep='\t')
fungi_ASV = pd.read_csv("/mnt/d/study/master/meiji/fungi_ASV.csv", sep=",", header=0)
# 构建要选择的列名列表，包括 "ASV" 和所有的样本ID
sample_ids = metadata['Sample ID'].tolist()
columns_to_select = ["ASV"] + sample_ids
# 从 fungi_ASV DataFrame 中选择指定的列
f_ASV = fungi_ASV[columns_to_select]
# 将 'ASV' 列设置为 DataFrame 的索引 (行名)
f_ASV.set_index(keys="ASV", inplace=True)
f_tax = fungi_ASV.iloc[:, 1:9].copy()
f_tax.set_index(keys="ASV", inplace=True)
print(len(f_ASV))
# 计算 ASV 相对丰度
f_ASV_df = f_ASV.copy()

#按种
f_species = f_ASV.join(f_tax['Species'])
# 按照 'Species' 列进行分组，并对每个样本的丰度求和
f_species = f_species.groupby('Species')[sample_ids].sum()
#排序
f_species = f_species.sort_index()
# 将当前的 ASV 索引重置为一个普通列。
f_tax_reset_index = f_tax.reset_index()
# 将 'Species' 列设置为新的索引。
f_tax_species = f_tax_reset_index.groupby('Species').first()
f_tax_species = f_tax_species.drop(columns=['ASV'])
f_tax_species = f_tax_species.sort_index()
# 进行替换操作
f_tax_species['Phylum'] = f_tax_species['Phylum'].replace('p__unclassified_k__Fungi', 'Unclassified')
# 去除 'p__' 前缀
f_tax_species['Phylum'] = f_tax_species['Phylum'].astype(str).str.replace("p__", "")
# 获取所有唯一的物种学名
f_unique_species_names = f_species.index.unique()
# 创建一个从物种学名到 'species_数字' 格式的映射字典
f_species_to_numeric_id = {
    species_name: f"species_{idx + 1}"
    for idx, species_name in enumerate(f_unique_species_names.sort_values()) # 先排序确保一致性
}
# 转换 f_species 的索引
f_species = f_species.rename(index= f_species_to_numeric_id)
# 转换 f_tax_species 的索引
f_tax_species = f_tax_species.rename(index= f_species_to_numeric_id)
f_species_df = f_species.copy()
print(len(f_species_df))

#核心species
# 在 Python 中，可以直接用 DataFrame 的除法功能，对列求和然后进行广播除法
f_species_rel = f_species_df.div(f_species_df.sum(axis=0), axis=1)
# 先把样本、origin 和 species 丰度组合成一个长表
# pandas 的 stack() 和 reset_index() 可以实现类似 melt 的功能
f_rel_long = f_species_rel.stack().reset_index()
f_rel_long.columns = ['species', 'Sample', 'value'] # 重命名列以匹配 R 的 melt 结果
# 创建从 Sample ID 到 Origin 和 Niche 的映射字典
sample_origins_map = metadata.set_index('Sample ID')['Origin'].to_dict()
sample_niches_map = metadata.set_index('Sample ID')['Niche'].to_dict()
# 将信息添加到长表中
f_rel_long['Origin'] = f_rel_long['Sample'].map(sample_origins_map)
f_rel_long['Niche'] = f_rel_long['Sample'].map(sample_niches_map)
f_species_total_counts = f_species.sum(axis=1) # 计算每个 species 在所有样本中的总丰度
f_total_counts_across_all_species = f_species_total_counts.sum() # 计算所有 species 的总丰度
f_species_relative_abundance = f_species_total_counts / f_total_counts_across_all_species
# 定义整体相对丰度的阈值 (0.1% 转换为小数)
overall_abundance_threshold = 0.001
# 筛选出满足整体丰度阈值的 species 名称列表
f_species_above_overall_threshold = f_species_relative_abundance[f_species_relative_abundance >= overall_abundance_threshold].index.tolist()
# 现在，只保留那些通过了整体丰度筛选的 species，用于后续的居群出现次数计算
f_rel_long_filtered_by_overall_abundance = f_rel_long[f_rel_long['species'].isin(f_species_above_overall_threshold)]
#计算 species 在每个 Origin 中是否出现 (平均丰度 > 0) 
f_species_origin_mean_filtered = f_rel_long_filtered_by_overall_abundance.groupby(['species', 'Origin'])['value'].mean().reset_index()
f_species_origin_mean_filtered.rename(columns={'value': 'mean_abundance_in_origin'}, inplace=True)
# species 在某个 Origin 中出现，如果其在该 Origin 中的平均丰度大于 0
f_species_in_origin = f_species_origin_mean_filtered[f_species_origin_mean_filtered['mean_abundance_in_origin'] > 0]
f_species_origin_count = f_species_in_origin.groupby('species')['Origin'].count().reset_index(name='origin_count')
#计算 species 在每个 Niche 中是否出现 (平均丰度 > 0) 
f_species_niche_mean_filtered = f_rel_long_filtered_by_overall_abundance.groupby(['species', 'Niche'])['value'].mean().reset_index()
f_species_niche_mean_filtered.rename(columns={'value': 'mean_abundance_in_niche'}, inplace=True)
# species 在某个 Niche 中出现，如果其在该 Niche 中的平均丰度大于 0
f_species_in_niche = f_species_niche_mean_filtered[f_species_niche_mean_filtered['mean_abundance_in_niche'] > 0]
f_species_niche_count = f_species_in_niche.groupby('species')['Niche'].count().reset_index(name='niche_count')
# 筛选核心 species 
# 合并 Origin 和 Niche 出现次数
f_species_counts_combined = pd.merge(f_species_origin_count, f_species_niche_count, on='species', how='inner')
# 定义出现次数阈值
origin_occurrence_threshold = 3
niche_occurrence_threshold = 2
# 筛选满足所有条件的 species
f_core_species_names = f_species_counts_combined[
    (f_species_counts_combined['origin_count'] >= origin_occurrence_threshold) &
    (f_species_counts_combined['niche_count'] >= niche_occurrence_threshold)
]['species'].tolist()
# 提取核心 species 的相对丰度数据
f_core_abundance = f_species_rel.loc[f_core_species_names]
f_core_species_raw_counts = f_species.loc[f_core_species_names]
print(len(f_core_species_raw_counts))

#最特殊species
#计算所有 species 整体相对丰度的中位数 
f_median_relative_abundance = f_species_relative_abundance.median()
print(f_median_relative_abundance)
# 定义稀有类群的丰度阈值 
f_rare_threshold = 0.000008
# 筛选出稀有 species 
f_rare_species = f_species_relative_abundance[f_species_relative_abundance < f_rare_threshold]
# 获取稀有 species 的名称（索引）
f_rare_species_names = f_rare_species.index.tolist()
len(f_rare_species_names)
f_species_df_transposed = f_species_df.T 
f_speciesdfmelt = f_species_df_transposed.stack().reset_index()# 长宽表格转换
f_speciesdfmelt.columns = ['Sample', 'species', 'value']
f_speciesdfmelt = f_speciesdfmelt[f_speciesdfmelt['value'] > 0]
sample_group_map = metadata.set_index('Sample ID')['Group'].to_dict()
f_control_species = f_speciesdfmelt.copy()
f_control_species['Group'] = f_control_species['Sample'].map(sample_group_map)# 添加 Group 信息
f_control_species.dropna(subset=['Group'], inplace=True) # 移除 Group 为 NaN 的行
# 提取 Group 和 species 列并去重
f_controlspecies_unique = f_control_species[['Group', 'species']].drop_duplicates()
# 统计每个 species 出现在多少个 Group 中
f_controlspecies_unique = f_controlspecies_unique.groupby('species').size().reset_index(name='group_count') 
# 只保留那些只在一个 Group 中出现的 species
f_controlspecies_unique = f_controlspecies_unique[f_controlspecies_unique['group_count'] == 1] 
# 获取只在一个 Group 中出现的 species 名称列表
f_unique_species_names_single_group = f_controlspecies_unique['species'].tolist()
# 将两个列表转换为集合，以便快速找到交集
f_set_rare_species = set(f_rare_species_names)
f_set_unique_species_single_group = set(f_unique_species_names_single_group) # 使用这个变量
# 找到同时满足两个条件的 species (即两个集合的交集)
f_rare_unique_species_names = list(f_set_rare_species.intersection(f_set_unique_species_single_group))
#提取这些“最特殊稀有 species”的原始丰度数据
f_rare_unique_abundance = f_species_df.loc[f_rare_unique_species_names]
f_rare_unique_species_raw_counts = f_species.loc[f_rare_unique_species_names]
print(len(f_rare_unique_species_raw_counts))

#生态位广适类群和专适类群
print(" 开始生态位广适/专适类群识别 ")
# 执行生态位宽度置换检验与 Z-score 计算
num_permutations = 1000 
f_niche_breadth_results = classify_generalist_specialist(
    f_species_rel, num_permutations=num_permutations
)
print("\n 生态位宽度置换检验结果 (含 Z-score, 前5个 species) ")
print(f_niche_breadth_results.head())
# 筛选广适类群 (Generalist)及专适类群 (Specialist)
f_generalists_species = f_niche_breadth_results[f_niche_breadth_results['Ecotype'] == 'Generalist'].index.tolist()
f_specialists_species = f_niche_breadth_results[f_niche_breadth_results['Ecotype'] == 'Specialist'].index.tolist()
print(f"\n识别出 **{len(f_generalists_species)}** 个广适类群 species (Z-score >= 2)。")
print(f"识别出 **{len(f_specialists_species)}** 个专适类群 species (Z-score <= -2)。")
print("\n 广适类群 species 的相对丰度数据 (前5行，前5列) ")
if not f_generalists_species:
    print("无广适类群 species 数据。")
else:
    f_generalists_raw_counts = f_species.loc[f_generalists_species]
    f_generalists_abundance = f_species_rel.loc[f_generalists_species]
    print(f_generalists_abundance.head())
print("\n 专适类群 species 的相对丰度数据 (前5行，前5列) ")
if not f_specialists_species:
    print("无专适类群 species 数据。")
else:
    f_specialists_raw_counts = f_species.loc[f_specialists_species]
    f_specialists_abundance = f_species_rel.loc[f_specialists_species]
print(f_specialists_abundance.head())
print("\n 生态位广适/专适类群识别完成 ")

# 加载相对丰度、丰度物种、稀有物种、广适种和狭适种数据
f_df_OTU_rel = f_species_rel 
f_df_abun = f_core_abundance 
f_df_rare = f_rare_unique_abundance
f_df_gen = f_generalists_abundance
f_df_spe = f_specialists_abundance
# 加载生态系统数据
f_ecosystem = metadata
f_ecosystem = f_ecosystem.iloc[:, :2]
f_ecosystem.set_index(f_ecosystem.columns[0], inplace=True)
# 绘制分组的平均相对丰度 vs. 占据的位点数量的散点图
# 准备用于散点图的输入 DataFrame，包括平均相对丰度、占据的位点数量和分组
# 获取丰度物种、稀有物种、广适种和狭适种的OTU索引列表
f_abun_OTU = list(f_df_abun.index)
f_rare_OTU = list(f_df_rare.index)
f_gen_OTU = list(f_df_gen.index)
f_spe_OTU = list(f_df_spe.index)
# 计算每个OTU的平均相对丰度
# axis=1 表示对每一行（即每个OTU）进行操作
f_relabun = f_df_OTU_rel.mean(axis=1)
# 将 Series 转换为 DataFrame，并命名列为 'meanRelabun'
f_relabun = f_relabun.to_frame(name='meanRelabun')
# 计算每个OTU占据的位点数量（即在多少个样本中丰度不为0）
# iloc[:, 1:] 假设第一列是索引或OTU名称，我们只计算样本列
# != 0 检查每个值是否不为0，返回布尔值DataFrame
# sum(axis=1) 计算每行（每个OTU）的True（即不为0的位点）的数量
f_sites = (f_df_OTU_rel.iloc[:, 1:] != 0).sum(axis=1).to_frame(name='sitesCount')
# 将平均相对丰度 DataFrame 和位点数量 DataFrame 根据索引合并
f_relabun_sites = pd.merge(f_relabun, f_sites, left_index=True, right_index=True)
# 定义一个字典，将分组标签与其对应的OTU索引列表关联起来
f_lists = {'Cores': f_abun_OTU,
         'Uniques': f_rare_OTU,
         'Generalists': f_gen_OTU,
         'Specialists': f_spe_OTU}
# 在 f_relabun_sites DataFrame 中添加一个名为 'group' 的新列，用于存储分组标签
f_relabun_sites['group'] = ''
# 遍历定义的分组和其OTU索引列表
for label, indices in f_lists.items():
    # 根据OTU索引判断，将对应的分组标签赋给 'group' 列
    f_relabun_sites.loc[f_relabun_sites.index.isin(indices), 'group'] = label
# 过滤掉 'group' 列为空（即不属于任何已知分组）的行
f_relabun_sites = f_relabun_sites[f_relabun_sites['group'].astype(bool)]
# 按照自定义的顺序对 'group' 列进行排序，确保图例和绘图的顺序一致
f_custom_order = ["Cores", "Uniques", "Generalists", "Specialists"]
f_relabun_sites['group'] = pd.Categorical(f_relabun_sites['group'], categories= f_custom_order, ordered=True)
f_relabun_sites_sorted = f_relabun_sites.sort_values(by='group')
# 创建一个散点图
# make a scatter plot
# 定义分组与颜色之间的映射关系
f_group_colors = {
    'Cores': '#5499C7',  # 蓝色
    'Uniques': '#D4AC0D',      # 金色
    'Generalists': '#EC7063',    # 红色
    'Specialists': '#45B39D'     # 绿色
}
# 使用 seaborn 库创建散点图
# data: 要绘图的 DataFrame
# x: X轴数据 ('sitesCount' - 占据的位点数量)
# y: Y轴数据 ('meanRelabun' - 平均相对丰度)
# hue: 根据 'group' 列的值给散点上色，并自动生成图例
# palette: 使用预定义的颜色映射
# s: 散点的大小
# linewidth: 散点边缘线的宽度
# edgecolor: 散点边缘线的颜色
# alpha: 散点的透明度
f_ax = sns.scatterplot(data= f_relabun_sites_sorted, x='sitesCount', y='meanRelabun', hue='group',
                     palette= f_group_colors, s=100, linewidth=0.5, edgecolor='black', alpha=0.6)
# 设置图表的整体尺寸 (宽度, 高度)
plt.rcParams["figure.figsize"] = (4, 4) # 注意：plt.rcParams 应在 plt.figure() 或 sns.scatterplot() 之前设置，
                                     # 或者使用 plt.figure(figsize=(4,4)) 来创建图
# 将Y轴（平均相对丰度）设置为对数刻度，以便更好地展示大范围的丰度数据
plt.yscale('log')
# 设置X轴和Y轴的标签及其字体大小
f_ax.set_xlabel('Samples occupied', size=13)
f_ax.set_ylabel('Mean relative abundance', size=13)
# 设置图表的标题及其字体大小
f_ax.set_title('Fungal ecotype distribution', size=13)
# 显示图例 (由 hue 参数自动生成)
f_ax.legend()
# 定义输出目录（确保该目录存在，如果不存在，os.makedirs 会创建它）
output_dir = '/mnt/d/study/master/Extended_Data_tables/Figure_3/'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"创建输出目录: {output_dir}")
# 保存图表
plt.savefig(os.path.join(output_dir, 'ecotype_distribution_fungi.png'), bbox_inches='tight', dpi=600)
plt.show()

#  数据准备：不同分组在生态系统中的分布 
# 将各个分组（丰度物种、稀有物种、广适种、狭适种）的DataFrame进行转置
# 转置后，行是样本，列是OTU
f_df_abun_T = f_df_abun.T
# 计算每个样本中（即每行）非零OTU的数量（即该样本包含的该分组OTU的数量）
f_df_abun_ct = f_df_abun_T.apply(lambda row: np.count_nonzero(row), axis=1).to_frame(name='Count')
# 为这些计数添加一个 'group' 列，标记为 'Cores'
f_df_abun_ct['ecotype'] = 'Cores'
# 将计数DataFrame与生态系统元数据合并，基于它们的索引（样本ID）
f_df_abun_eco = pd.merge(f_df_abun_ct, f_ecosystem, left_index = True, right_index = True)
# 对稀有物种重复上述步骤
f_df_rare_T = f_df_rare.T
f_df_rare_ct = f_df_rare_T.apply(lambda row: np.count_nonzero(row), axis=1).to_frame(name='Count')
f_df_rare_ct['ecotype'] = 'Uniques'
f_df_rare_eco = pd.merge(f_df_rare_ct, f_ecosystem, left_index = True, right_index = True)
# 对广适种重复上述步骤
f_df_gen_T = f_df_gen.T
f_df_gen_ct = f_df_gen_T.apply(lambda row: np.count_nonzero(row), axis=1).to_frame(name='Count')
f_df_gen_ct['ecotype'] = 'Generalists'
f_df_gen_eco = pd.merge(f_df_gen_ct, f_ecosystem, left_index = True, right_index = True)
# 对狭适种重复上述步骤
f_df_spe_T = f_df_spe.T
f_df_spe_ct = f_df_spe_T.apply(lambda row: np.count_nonzero(row), axis=1).to_frame(name='Count')
f_df_spe_ct['ecotype'] = 'Specialists'
f_df_spe_eco = pd.merge(f_df_spe_ct, f_ecosystem, left_index = True, right_index = True)
# 将所有分组的数据（丰度物种、稀有物种、广适种、狭适种）纵向合并到一个大的DataFrame中
f_df_ecotype_eco = pd.concat([f_df_abun_eco, f_df_rare_eco, f_df_gen_eco, f_df_spe_eco])
#  绘图参数设置 
# 定义箱线图的X轴、Y轴和分组（颜色区分）变量
x = "ecotype" # X轴：分组
y = "Count" # Y轴：OTU数量
hue = "Group" # 分组（颜色）：生态系统类型
# 定义 hue 变量（生态系统类型）的显示顺序，以便在图例和绘图中保持一致
f_hue_order = ["JRG", "JJG", "TZG", "PAG", "JRN", "JJN", "TZN", "PAN"]
# 定义 X 轴上分组（group）的显示顺序
f_order = ["Cores", "Uniques", "Generalists", "Specialists"]
# 定义生态系统类型与颜色之间的映射关系
f_group_colors = {
   "JRG": "#8C57A2FF",
   "JJG": "#3EBCB6",
   "TZG": "#82581FFF",
   "PAG": "#2F509EFF",
   "JRN": "#E5614CFF",
   "JJN": "#97A1A7FF",
   "TZN": "#DC9445FF",
   "PAN": "#bee183"
}
# 定义要进行统计比较的配对列表
# 每个元组包含两个要比较的组，格式为 ((group_x, hue_x), (group_y, hue_y))
# 这些是在FDR校正后仍然显著的比较对，未显著的已被注释掉
pairs = [
    # Cores (核心物种) 的比较对 (共28对)
    (("Cores", "JRG"), ("Cores", "JJG")),
    (("Cores", "JRG"), ("Cores", "TZG")),
    (("Cores", "JRG"), ("Cores", "PAG")),
    (("Cores", "JRG"), ("Cores", "JRN")),
    (("Cores", "JRG"), ("Cores", "JJN")),
    (("Cores", "JRG"), ("Cores", "TZN")),
    (("Cores", "JRG"), ("Cores", "PAN")),
    (("Cores", "JJG"), ("Cores", "TZG")),
    (("Cores", "JJG"), ("Cores", "PAG")),
    (("Cores", "JJG"), ("Cores", "JRN")),
    (("Cores", "JJG"), ("Cores", "JJN")),
    (("Cores", "JJG"), ("Cores", "TZN")),
    (("Cores", "JJG"), ("Cores", "PAN")),
    (("Cores", "TZG"), ("Cores", "PAG")),
    (("Cores", "TZG"), ("Cores", "JRN")),
    (("Cores", "TZG"), ("Cores", "JJN")),
    (("Cores", "TZG"), ("Cores", "TZN")),
    (("Cores", "TZG"), ("Cores", "PAN")),
    (("Cores", "PAG"), ("Cores", "JRN")),
    (("Cores", "PAG"), ("Cores", "JJN")),
    (("Cores", "PAG"), ("Cores", "TZN")),
    (("Cores", "PAG"), ("Cores", "PAN")),
    (("Cores", "JRN"), ("Cores", "JJN")),
    (("Cores", "JRN"), ("Cores", "TZN")),
    (("Cores", "JRN"), ("Cores", "PAN")),
    (("Cores", "JJN"), ("Cores", "TZN")),
    (("Cores", "JJN"), ("Cores", "PAN")),
    (("Cores", "TZN"), ("Cores", "PAN")),

    # Uniques (稀有物种) 的比较对 (共28对)
    (("Uniques", "JRG"), ("Uniques", "JJG")),
    (("Uniques", "JRG"), ("Uniques", "TZG")),
    (("Uniques", "JRG"), ("Uniques", "PAG")),
    (("Uniques", "JRG"), ("Uniques", "JRN")),
    (("Uniques", "JRG"), ("Uniques", "JJN")),
    (("Uniques", "JRG"), ("Uniques", "TZN")),
    (("Uniques", "JRG"), ("Uniques", "PAN")),
    (("Uniques", "JJG"), ("Uniques", "TZG")),
    (("Uniques", "JJG"), ("Uniques", "PAG")),
    (("Uniques", "JJG"), ("Uniques", "JRN")),
    (("Uniques", "JJG"), ("Uniques", "JJN")),
    (("Uniques", "JJG"), ("Uniques", "TZN")),
    (("Uniques", "JJG"), ("Uniques", "PAN")),
    (("Uniques", "TZG"), ("Uniques", "PAG")),
    (("Uniques", "TZG"), ("Uniques", "JRN")),
    (("Uniques", "TZG"), ("Uniques", "JJN")),
    (("Uniques", "TZG"), ("Uniques", "TZN")),
    (("Uniques", "TZG"), ("Uniques", "PAN")),
    (("Uniques", "PAG"), ("Uniques", "JRN")),
    (("Uniques", "PAG"), ("Uniques", "JJN")),
    (("Uniques", "PAG"), ("Uniques", "TZN")),
    (("Uniques", "PAG"), ("Uniques", "PAN")),
    (("Uniques", "JRN"), ("Uniques", "JJN")),
    (("Uniques", "JRN"), ("Uniques", "TZN")),
    (("Uniques", "JRN"), ("Uniques", "PAN")),
    (("Uniques", "JJN"), ("Uniques", "TZN")),
    (("Uniques", "JJN"), ("Uniques", "PAN")),
    (("Uniques", "TZN"), ("Uniques", "PAN")),

    # Generalists (广适种) 的比较对 (共28对)
    (("Generalists", "JRG"), ("Generalists", "JJG")),
    (("Generalists", "JRG"), ("Generalists", "TZG")),
    (("Generalists", "JRG"), ("Generalists", "PAG")),
    (("Generalists", "JRG"), ("Generalists", "JRN")),
    (("Generalists", "JRG"), ("Generalists", "JJN")),
    (("Generalists", "JRG"), ("Generalists", "TZN")),
    (("Generalists", "JRG"), ("Generalists", "PAN")),
    (("Generalists", "JJG"), ("Generalists", "TZG")),
    (("Generalists", "JJG"), ("Generalists", "PAG")),
    (("Generalists", "JJG"), ("Generalists", "JRN")),
    (("Generalists", "JJG"), ("Generalists", "JJN")),
    (("Generalists", "JJG"), ("Generalists", "TZN")),
    (("Generalists", "JJG"), ("Generalists", "PAN")),
    (("Generalists", "TZG"), ("Generalists", "PAG")),
    (("Generalists", "TZG"), ("Generalists", "JRN")),
    (("Generalists", "TZG"), ("Generalists", "JJN")),
    (("Generalists", "TZG"), ("Generalists", "TZN")),
    (("Generalists", "TZG"), ("Generalists", "PAN")),
    (("Generalists", "PAG"), ("Generalists", "JRN")),
    (("Generalists", "PAG"), ("Generalists", "JJN")),
    (("Generalists", "PAG"), ("Generalists", "TZN")),
    (("Generalists", "PAG"), ("Generalists", "PAN")),
    (("Generalists", "JRN"), ("Generalists", "JJN")),
    (("Generalists", "JRN"), ("Generalists", "TZN")),
    (("Generalists", "JRN"), ("Generalists", "PAN")),
    (("Generalists", "JJN"), ("Generalists", "TZN")),
    (("Generalists", "JJN"), ("Generalists", "PAN")),
    (("Generalists", "TZN"), ("Generalists", "PAN")),

    # Specialists (狭适种) 的比较对 (共28对)
    (("Specialists", "JRG"), ("Specialists", "JJG")),
    (("Specialists", "JRG"), ("Specialists", "TZG")),
    (("Specialists", "JRG"), ("Specialists", "PAG")),
    (("Specialists", "JRG"), ("Specialists", "JRN")),
    (("Specialists", "JRG"), ("Specialists", "JJN")),
    (("Specialists", "JRG"), ("Specialists", "TZN")),
    (("Specialists", "JRG"), ("Specialists", "PAN")),
    (("Specialists", "JJG"), ("Specialists", "TZG")),
    (("Specialists", "JJG"), ("Specialists", "PAG")),
    (("Specialists", "JJG"), ("Specialists", "JRN")),
    (("Specialists", "JJG"), ("Specialists", "JJN")),
    (("Specialists", "JJG"), ("Specialists", "TZN")),
    (("Specialists", "JJG"), ("Specialists", "PAN")),
    (("Specialists", "TZG"), ("Specialists", "PAG")),
    (("Specialists", "TZG"), ("Specialists", "JRN")),
    (("Specialists", "TZG"), ("Specialists", "JJN")),
    (("Specialists", "TZG"), ("Specialists", "TZN")),
    (("Specialists", "TZG"), ("Specialists", "PAN")),
    (("Specialists", "PAG"), ("Specialists", "JRN")),
    (("Specialists", "PAG"), ("Specialists", "JJN")),
    (("Specialists", "PAG"), ("Specialists", "TZN")),
    (("Specialists", "PAG"), ("Specialists", "PAN")),
    (("Specialists", "JRN"), ("Specialists", "JJN")),
    (("Specialists", "JRN"), ("Specialists", "TZN")),
    (("Specialists", "JRN"), ("Specialists", "PAN")),
    (("Specialists", "JJN"), ("Specialists", "TZN")),
    (("Specialists", "JJN"), ("Specialists", "PAN")),
    (("Specialists", "TZN"), ("Specialists", "PAN"))
]
#  提取 P 值并执行 FDR 校正 
# 存储原始 P 值
p_values = []
# 存储每个 P 值对应的比较对，以便后续追踪
comparison_labels = []
# 对每一对进行循环
for pair in pairs:
    ecotype1, hue1 = pair[0]
    ecotype2, hue2 = pair[1]
    # 从 DataFrame 中筛选出第一组的数据
    data1 = f_df_ecotype_eco[(f_df_ecotype_eco['ecotype'] == ecotype1) &
                           (f_df_ecotype_eco['Group'] == hue1)]['Count']
    # 从 DataFrame 中筛选出第二组的数据
    data2 = f_df_ecotype_eco[(f_df_ecotype_eco['ecotype'] == ecotype2) &
                           (f_df_ecotype_eco['Group'] == hue2)]['Count']
    # 确保两组都有数据才能进行比较
    if len(data1) > 1 and len(data2) > 1: # 通常需要至少2个样本才能进行T检验
        # 执行 Mann-Whitney U 检验（非参数，对数据分布没有严格要求）
        # 如果您确定数据服从正态分布且方差齐性，可以使用 stats.ttest_ind
        statistic, p_val = stats.mannwhitneyu(data1, data2, alternative='two-sided')
        p_values.append(p_val)
        comparison_labels.append(f"{ecotype1}-{hue1} vs {ecotype2}-{hue2}")
    else:
        # 如果数据不足，则跳过或记录为 NaN
        p_values.append(np.nan)
        comparison_labels.append(f"{ecotype1}-{hue1} vs {ecotype2}-{hue2} (数据不足)")
# 过滤掉由于数据不足而产生的 NaN P 值
valid_p_values = [p for p in p_values if not np.isnan(p)]
valid_comparison_labels = [label for p, label in zip(p_values, comparison_labels) if not np.isnan(p)]
if len(valid_p_values) == 0:
    print("没有足够的有效比较可以执行 FDR 校正。请检查您的数据和比较对。")
else:
    # 执行 FDR 校正 (Benjamini-Hochberg 方法)
    # alpha 是您设定的显著性水平，例如 0.05
    reject, pvals_corrected = fdrcorrection(pvals=valid_p_values, alpha=0.05, method='indep')
    # 结果展示 
    print("\n FDR 校正结果 (Benjamini-Hochberg) ")
    print(f"原始有效比较数量: {len(valid_p_values)}")
    print(f"FDR 校正后的显著性水平 (alpha): 0.05")
    significant_pairs_after_fdr = []
    insignificant_pairs_after_fdr = []
    for i, label in enumerate(valid_comparison_labels):
        original_p = valid_p_values[i]
        corrected_p = pvals_corrected[i]
        is_significant = reject[i]
        result_string = f"比较: {label}, 原始 P 值: {original_p:.4f}, 校正后 P 值 (q 值): {corrected_p:.4f}, 是否显著: {is_significant}"
        print(result_string)
        if is_significant:
            significant_pairs_after_fdr.append(label)
        else:
            insignificant_pairs_after_fdr.append(label)
    print("\n 显著的比较对 (FDR 校正后) ")
    if significant_pairs_after_fdr:
        for p in significant_pairs_after_fdr:
            print(p)
    else:
        print("FDR 校正后没有发现显著的比较对。")
    print("\n 不显著的比较对 (FDR 校正后) ")
    if insignificant_pairs_after_fdr:
        for p in insignificant_pairs_after_fdr:
            print(p)
    else:
        print("所有比较对在 FDR 校正后均显著。")
f_pairs = [
    # Cores (核心物种) 的显著比较对
    (("Cores", "JRG"), ("Cores", "TZG")),
    (("Cores", "JRG"), ("Cores", "JRN")),
    (("Cores", "JRG"), ("Cores", "JJN")),
    (("Cores", "JRG"), ("Cores", "TZN")),
    (("Cores", "JRG"), ("Cores", "PAN")),
    (("Cores", "JJG"), ("Cores", "JRN")),
    (("Cores", "JJG"), ("Cores", "JJN")),
    (("Cores", "JJG"), ("Cores", "TZN")),
    (("Cores", "JJG"), ("Cores", "PAN")),
    (("Cores", "TZG"), ("Cores", "PAG")),
    (("Cores", "TZG"), ("Cores", "JRN")),
    (("Cores", "TZG"), ("Cores", "JJN")),
    (("Cores", "TZG"), ("Cores", "TZN")),
    (("Cores", "TZG"), ("Cores", "PAN")),
    (("Cores", "PAG"), ("Cores", "JRN")),
    (("Cores", "PAG"), ("Cores", "JJN")),
    (("Cores", "PAG"), ("Cores", "TZN")),
    (("Cores", "PAG"), ("Cores", "PAN")),
    (("Cores", "JRN"), ("Cores", "PAN")),
    (("Cores", "TZN"), ("Cores", "PAN")),

    # Uniques (稀有物种) 的显著比较对
    (("Uniques", "JRG"), ("Uniques", "JJG")),
    (("Uniques", "JRG"), ("Uniques", "TZG")),
    (("Uniques", "JRG"), ("Uniques", "PAG")),
    (("Uniques", "JRG"), ("Uniques", "JRN")),
    (("Uniques", "JRG"), ("Uniques", "JJN")),
    (("Uniques", "JRG"), ("Uniques", "TZN")),
    (("Uniques", "JRG"), ("Uniques", "PAN")),
    (("Uniques", "JJG"), ("Uniques", "JRN")),
    (("Uniques", "JJG"), ("Uniques", "JJN")),
    (("Uniques", "JJG"), ("Uniques", "TZN")),
    (("Uniques", "JJG"), ("Uniques", "PAN")),
    (("Uniques", "TZG"), ("Uniques", "PAG")),
    (("Uniques", "TZG"), ("Uniques", "JRN")),
    (("Uniques", "TZG"), ("Uniques", "JJN")),
    (("Uniques", "TZG"), ("Uniques", "TZN")),
    (("Uniques", "TZG"), ("Uniques", "PAN")),
    (("Uniques", "PAG"), ("Uniques", "JRN")),
    (("Uniques", "PAG"), ("Uniques", "JJN")),
    (("Uniques", "PAG"), ("Uniques", "TZN")),
    (("Uniques", "PAG"), ("Uniques", "PAN")),

    # Generalists (广适种) 的显著比较对
    (("Generalists", "JRG"), ("Generalists", "JJG")),
    (("Generalists", "JRG"), ("Generalists", "TZG")),
    (("Generalists", "JRG"), ("Generalists", "PAG")),
    (("Generalists", "JRG"), ("Generalists", "JRN")),
    (("Generalists", "JRG"), ("Generalists", "JJN")),
    (("Generalists", "JRG"), ("Generalists", "TZN")),
    (("Generalists", "JRG"), ("Generalists", "PAN")),
    (("Generalists", "JJG"), ("Generalists", "JRN")),
    (("Generalists", "JJG"), ("Generalists", "JJN")),
    (("Generalists", "JJG"), ("Generalists", "TZN")),
    (("Generalists", "JJG"), ("Generalists", "PAN")),
    (("Generalists", "TZG"), ("Generalists", "JRN")),
    (("Generalists", "TZG"), ("Generalists", "JJN")),
    (("Generalists", "TZG"), ("Generalists", "TZN")),
    (("Generalists", "TZG"), ("Generalists", "PAN")),
    (("Generalists", "PAG"), ("Generalists", "JRN")),
    (("Generalists", "PAG"), ("Generalists", "JJN")),
    (("Generalists", "PAG"), ("Generalists", "TZN")),
    (("Generalists", "PAG"), ("Generalists", "PAN")),

    # Specialists (狭适种) 的显著比较对
    (("Specialists", "JRG"), ("Specialists", "JJG")),
    (("Specialists", "JRG"), ("Specialists", "TZG")),
    (("Specialists", "JRG"), ("Specialists", "PAG")),
    (("Specialists", "JRG"), ("Specialists", "JRN")),
    (("Specialists", "JRG"), ("Specialists", "JJN")),
    (("Specialists", "JRG"), ("Specialists", "TZN")),
    (("Specialists", "JRG"), ("Specialists", "PAN")),
    (("Specialists", "JJG"), ("Specialists", "TZG")),
    (("Specialists", "JJG"), ("Specialists", "JRN")),
    (("Specialists", "JJG"), ("Specialists", "JJN")),
    (("Specialists", "JJG"), ("Specialists", "TZN")),
    (("Specialists", "JJG"), ("Specialists", "PAN")),
    (("Specialists", "TZG"), ("Specialists", "PAG")),
    (("Specialists", "TZG"), ("Specialists", "JRN")),
    (("Specialists", "TZG"), ("Specialists", "JJN")),
    (("Specialists", "TZG"), ("Specialists", "TZN")),
    (("Specialists", "TZG"), ("Specialists", "PAN")),
    (("Specialists", "PAG"), ("Specialists", "JRN")),
    (("Specialists", "PAG"), ("Specialists", "JJN")),
    (("Specialists", "PAG"), ("Specialists", "TZN")),
    (("Specialists", "PAG"), ("Specialists", "PAN"))
]
# 设置 Matplotlib 图的尺寸（宽度，高度）
plt.rcParams["figure.figsize"] = (6, 6)
# 使用 Seaborn 绘制箱线图 (Box Plot)
f_ax = sns.boxplot(data=f_df_ecotype_eco, # 输入数据 DataFrame
                 x=x,               # X轴变量
                 y=y,               # Y轴变量
                 order=f_order,       # X轴分类的显示顺序
                 hue=hue,           # 根据哪个变量进行分组并上色
                 hue_order= f_hue_order, # hue变量的显示顺序
                 palette=f_group_colors,
                 legend=False) # 使用预定义的颜色映射
# 初始化 Annotator 对象，用于添加统计显著性注释
# 传入与 boxplot 相同的绘图参数，确保注释正确对齐
annot = Annotator(f_ax, f_pairs, data=f_df_ecotype_eco, x=x, y=y, order=f_order, hue=hue, hue_order= f_hue_order)
# 配置注释的测试方法和P值校正
# test='Mann-Whitney': 使用 Mann-Whitney U 检验进行非参数比较
# comparisons_correction="Benjamini-Hochberg": 使用 Benjamini-Hochberg 方法进行多重检验校正（FDR校正）
# verbose=2: 显示详细的测试结果信息
annot.configure(test='Mann-Whitney', comparisons_correction="Benjamini-Hochberg", verbose=2)
# 应用统计检验
annot.apply_test()
# 在图上绘制显著性标记（例如星号）
annot.annotate()
# 设置X轴标签为空字符串，因为组名本身已经很清晰
f_ax.set_xlabel('')
# 设置Y轴标签及其字体大小
f_ax.set_ylabel('Count', size=13)
# 设置图表的标题及其字体大小
f_ax.set_title('Number of fungal species for ecotypes', size=13)
# 旋转X轴刻度标签，并调整字体大小和对齐方式，以防止重叠
plt.xticks(fontsize=13, rotation=45, ha='right')
# 调整图例的位置，使其位于图表外部的右侧中间位置
# title="Ecosystems": 设置图例标题
# bbox_to_anchor=(1.03, 0.35): 将图例锚点移动到轴外部 (1.03 表示X轴的103%位置，0.35 表示Y轴的35%位置)
#plt.legend(title="Ecosystems", bbox_to_anchor=(1.03, 0.35))
# 定义输出目录
output_dir = '/mnt/d/study/master/Extended_Data_tables/Figure_3/'
# 如果目录不存在，则创建它
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"已创建输出目录: {output_dir}")
# 保存图表
# bbox_inches='tight' 确保所有图表元素（包括图例和标签）都包含在保存的图像中
# dpi=600 设置图像的分辨率，提高清晰度
plt.savefig(os.path.join(output_dir, 'ecotype_ecosystems_fungi.png'), bbox_inches='tight', dpi=600)
# 显示图表
plt.show()

#  计算分组所代表的门（Phyla）的比例 
# 为每个分组计算其所包含的门（Phylum）的观察比例
# 提取属于“丰度物种”的OTU的分类学信息
f_taxon = f_tax_species
# 定义模糊（Phylum）的模式（不区分大小写）
ambiguous_patterns = re.compile(
   r"uncultured|unclassified|norank",
   flags=re.IGNORECASE
)
# 统一处理函数，包含去除前缀和过滤模糊门
def get_filtered_phylum_proportion(df_group, taxon_df, ambiguous_pat):
    # 筛选出索引（OTU ID）存在于 df_group 中的分类学信息
    taxon_filtered = taxon_df[taxon_df.index.isin(df_group.index)].copy() # 使用 .copy() 避免 SettingWithCopyWarning
    # 去除 'p__' 前缀
    taxon_filtered['Phylum'] = taxon_filtered['Phylum'].astype(str).str.replace("p__", "")
    # 过滤掉匹配模糊模式的门
    # 使用 apply 和 lambda 函数来检查每个门名是否包含模糊模式
    taxon_filtered = taxon_filtered[~taxon_filtered['Phylum'].apply(lambda x: bool(ambiguous_pat.search(str(x))))]
    # 计算 'Phylum' 列中每个门的出现频率，并将其归一化为比例
    prop = taxon_filtered['Phylum'].value_counts(normalize=True).to_frame(name='Proportion')
    return prop
# 计算每个分组的门比例
f_prop_abun = get_filtered_phylum_proportion(f_df_abun, f_taxon, ambiguous_patterns)
f_prop_rare = get_filtered_phylum_proportion(f_df_rare, f_taxon, ambiguous_patterns)
f_prop_gen = get_filtered_phylum_proportion(f_df_gen, f_taxon, ambiguous_patterns)
f_prop_spe = get_filtered_phylum_proportion(f_df_spe, f_taxon, ambiguous_patterns)
#  绘制条形图显示每个分组的门比例 
# 设置 Matplotlib 图的默认尺寸（宽度，高度）
plt.rcParams["figure.figsize"] = (4, 4)
# 定义一个函数用于绘制单个分组的门比例条形图
def barplot_prop(df, groupname, color):
    """
    绘制给定DataFrame中每个门的比例条形图。
    参数:
        df (DataFrame): 包含 'Proportion' 列和 Phylum 作为索引的 DataFrame。
        groupname (str): 图表的标题和保存文件名的一部分，代表分组名称。
        color (str): 用于条形图的颜色。
    """
    # 使用 seaborn 绘制条形图
    # x = df.index: X轴为门的名称（DataFrame的索引）
    # y = df['Proportion']: Y轴为该门的比例
    # color = color: 设置条形图的颜色
    ax = sns.barplot(x = df.index, y = df['Proportion'], color = color)
    # 设置X轴标签、Y轴标签和图表标题
    ax.set_xlabel(' Fungal phylum', size=13)
    ax.set_ylabel('Proportion', size=13)
    ax.set_title(groupname, size=13)
    # 旋转X轴刻度标签90度，并设置字体大小，防止标签重叠
    plt.xticks(fontsize=11, rotation = 90)
    # 定义保存图表的输出目录
    output_dir = '/mnt/d/study/master/Extended_Data_tables/Figure_3/'
    # 如果目录不存在，则创建它
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"已创建输出目录: {output_dir}")
    # 保存图表
    # bbox_inches='tight' 确保所有图表元素都包含在保存的图像中
    # dpi=600 设置图像的分辨率
    plt.savefig(os.path.join(output_dir, groupname + '_phylum_bar_fungi.png'), bbox_inches='tight', dpi=600)
    # 显示图表
    plt.show()
# 调用函数为每个分组生成并保存条形图
barplot_prop(f_prop_abun, 'Cores', '#5499C7')   # 丰度物种
barplot_prop(f_prop_rare, 'Uniques', '#D4AC0D')       # 稀有物种
barplot_prop(f_prop_gen, 'Generalists', '#EC7063')      # 广适种
barplot_prop(f_prop_spe, 'Specialists', '#45B39D')      # 狭适种
